In [ ]:
# setup
import cv2
import os
import sys
import pickle
import numpy as np

In [5]:
BASE_DIR = os.getcwd()  # the folder your .ipynb is in
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
MODEL_PATH = os.path.join(BASE_DIR, "trained_model.yml")
LABELS_PATH = os.path.join(BASE_DIR, "labels.pickle")

CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_detector = cv2.CascadeClassifier(CASCADE_PATH)

os.makedirs(DATASET_DIR, exist_ok=True)

if not hasattr(cv2, "face"):
    raise ImportError(
        "cv2.face not found. Run: pip uninstall opencv-python && "
        "pip install opencv-contrib-python, then RESTART the kernel."
    )
print("Setup OK. Dataset will be stored in:", DATASET_DIR)

Setup OK. Dataset will be stored in: C:\Users\parsh\face_detection\dataset


In [16]:
# dataset
def capture_dataset(name, num_samples=60):
    person_dir = os.path.join(DATASET_DIR, name)
    os.makedirs(person_dir, exist_ok=True)

    existing = [f for f in os.listdir(person_dir) if f.lower().endswith(".jpg")]
    count = len(existing)
    target = count + num_samples

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not access the webcam.")

    print(f"[CAPTURE] Collecting images for '{name}'. Press 'q' to stop early.")
    try:
        while count < target:
            ret, frame = cap.read()
            if not ret:
                continue

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_detector.detectMultiScale(
                gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80)
            )

            if len(faces) != 1:
                cv2.putText(frame, "Align ONE face in frame", (20, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            else:
                (x, y, w, h) = faces[0]
                face_crop = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
                count += 1
                cv2.imwrite(os.path.join(person_dir, f"{count}.jpg"), face_crop)
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                cv2.putText(frame, f"Saved {count}/{target}", (20, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.imshow("Dataset Capture - press 'q' to quit", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)

    print(f"[CAPTURE] Done. {count} total images stored for '{name}'.")

In [ ]:
capture_dataset("earbuds")

In [ ]:
# trainings
def train_recognizer():
    if not os.path.isdir(DATASET_DIR) or not os.listdir(DATASET_DIR):
        raise RuntimeError("No dataset found. Run capture_dataset() first.")

    faces, labels, label_map = [], [], {}
    current_label = 0

    for person_name in sorted(os.listdir(DATASET_DIR)):
        person_dir = os.path.join(DATASET_DIR, person_name)
        if not os.path.isdir(person_dir):
            continue
        images = [f for f in os.listdir(person_dir) if f.lower().endswith((".jpg", ".png"))]
        if not images:
            continue
        if len(images) < 10:
            print(f"[WARN] '{person_name}' has only {len(images)} images (recommend 20-30+).")

        label_map[current_label] = person_name
        for img_name in images:
            img = cv2.imread(os.path.join(person_dir, img_name), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            faces.append(img)
            labels.append(current_label)
        current_label += 1

    if not faces:
        raise RuntimeError("No valid training images found.")
    if len(label_map) < 2:
        print("[WARN] Only one person in dataset — recognition won't distinguish others well.")

    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.train(faces, np.array(labels))
    recognizer.save(MODEL_PATH)
    with open(LABELS_PATH, "wb") as f:
        pickle.dump(label_map, f)

    print(f"[TRAIN] Trained on {len(faces)} images, {len(label_map)} people: {list(label_map.values())}")

In [ ]:
train_recognizer()

In [14]:
# reconginization
def recognize_faces(confidence_threshold=70):
    if not os.path.exists(MODEL_PATH) or not os.path.exists(LABELS_PATH):
        raise RuntimeError("No trained model found. Run train_recognizer() first.")

    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.read(MODEL_PATH)
    with open(LABELS_PATH, "rb") as f:
        label_map = pickle.load(f)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not access the webcam.")

    greeted = set()
    print("[RECOGNIZE] Press 'q' to quit.")
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                continue
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))

            for (x, y, w, h) in faces:
                face_crop = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
                label_id, confidence = recognizer.predict(face_crop)

                if confidence < confidence_threshold and label_id in label_map:
                    name = label_map[label_id]
                    color = (0, 255, 0)
                    if name not in greeted:
                        print(f"Hello {name}!")
                        greeted.add(name)
                else:
                    name, color = "Unknown", (0, 0, 255)

                cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
                cv2.putText(frame, f"{name} ({confidence:.0f})", (x, y-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            cv2.imshow("Face Recognition - press 'q' to quit", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)

In [ ]:
recognize_faces()